# Assignment 1 — Tasks A–G

**Course:** Problem Solving With Unstructured Data and GenAI, MSBT&AI Fall 2026  

**Team members:** Stephen, Matthew, Rohan, Valentina

**Status:** Review draft with explicit Task A–G sections. The headings organize the work; they do not imply that unfinished analyses are complete.

## Using this notebook in Google Colab

Choose **File → Upload notebook** and select this file. Run the setup cell and upload `sample_data.csv` when prompted. The saved outputs come from the earlier audit; use **Runtime → Run all** to reproduce them. No API calls occur in this version.

Use Colab's table of contents to navigate directly to **Task A**, **Task B**, **Task C**, **Task D**, **Task E**, **Task F**, and **Task G**.

| Task | Current status |
|---|---|
| A — Discover entities and attributes | Audit and candidate tables present; discovery validation and final rankings remain |
| B — NLP competitive baseline | Tested lift function and selected pair results present; complete ten-brand matrix remains |
| C — LLM semantic lift and comparison | Method described; execution remains |
| D — MDS and clustering | Method described; execution remains |
| E — Brand–attribute positioning | Selected lexical examples present; validated five-by-five analysis remains |
| F — Most aspirational brand | Definition and required work described; analysis remains |
| G — Client recommendations | Required synthesis described; evidence-based final advice remains |

Read docs/SESSION_HANDOFF.md and the documents it links. Continue the grilling session at Task E, questions Q27–Q29. Keep recording finalized decisions.

The session handoff (/C:/Users/Rohan Verma/OneDrive/Desktop/MSITM/Problem Solving w GenAI/Assignment 1/docs/SESSION_HANDOFF.md) contains the unanswered Task E questions and distinguishes accepted decisions from remaining implementation details. All local documentation links were checked.


## Setup

In [ ]:
from pathlib import Path
import hashlib
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

# Set EDMUNDS_DATA only when using another local file path.
# In Colab, /content is the runtime's temporary file area.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

default_path = Path('/content/sample_data.csv') if IN_COLAB else Path('sample_data.csv')
DATA_PATH = Path(os.environ.get('EDMUNDS_DATA', str(default_path)))
if not DATA_PATH.is_file():
    if not IN_COLAB:
        raise FileNotFoundError('Place sample_data.csv beside this notebook, or set EDMUNDS_DATA.')
    print('Upload the course sample_data.csv file below.')
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(csv_names) != 1:
        raise ValueError('Please upload exactly one CSV, then rerun this cell.')
    DATA_PATH = Path(csv_names[0]).resolve()
    # Keep the selected upload path if the setup cell is run again.
    os.environ['EDMUNDS_DATA'] = str(DATA_PATH)

raw_bytes = DATA_PATH.read_bytes()
print('Input:', DATA_PATH.name)
print('Bytes:', len(raw_bytes))
print('SHA-256:', hashlib.sha256(raw_bytes).hexdigest())


Input: sample_data.csv
Bytes: 3391987
SHA-256: 1af81f6b88bec9ae0bf1cf668eaefdee86cc69b5ad27d3d74a797ab4a7991bad


# Task A — Discover the important entities and attributes

**Required:** Discover the ten most important brands and product attributes without a supplied brand/model dictionary. Reconcile model names, abbreviations and variants, define importance, report frequencies, and justify and validate the decisions.

**Current work:** Comprehensive audit and provisional candidate frequencies. Final discovery and semantic validation remain unfinished.

## Governing rubric and analytical commitments

Task A requires dictionary-free discovery of the ten most important automobile brands and important product attributes, including reconciliation of model names, abbreviations, variants, and other forms. The notebook must explain what was automated, what required semantic interpretation, how “important” was defined, and how results were validated.

For this audit:

- The **unit of analysis is a post**, not a raw token. Post prevalence is less vulnerable than token count to repeated words and quoted material.
- “Important” will ultimately mean **number and share of unique substantive posts that refer to an entity**, after entity resolution and removal of duplicates/quoted carryover. Raw mention counts are retained only as a diagnostic.
- The raw text is immutable. Cleaning decisions must produce additional columns and an audit trail.
- A simple lexical baseline must precede LLM interpretation.
- No LLM classification result is accepted without a stratified manual review and an error analysis.
- Every actual LLM run must log model, calls, input/output/total tokens, and estimated cost. No LLM API calls are made in this audit notebook.

## 1 Structure and schema

The file has no header row. Inspection of repeated value patterns supports the semantic labels `author`, `date_label`, and `text`. The date labels appear to encode year and month (for example, `6-Oct`), but this should be confirmed from data provenance; they are not full timestamps.

In [3]:
df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=["author", "date_label", "text"],
    dtype="string",
    keep_default_na=False,
)

structure = pd.DataFrame({
    "metric": ["rows", "columns", "unique authors", "unique date labels", "unique text values"],
    "value": [len(df), df.shape[1], df.author.nunique(), df.date_label.nunique(), df.text.nunique()],
})
display(structure)
display(df.head(5).assign(text=lambda x: x.text.str.slice(0, 240)))

,metric,value
0,rows,6000
1,columns,3
2,unique authors,329
3,unique date labels,44
4,unique text values,5986


,author,date_label,text
0,brightness04,6-Oct,""" kia hyundai under-achieving "" kia hyundai and "" kia hyundai over-achieving "" kia hyundai are relative terms . . . boy i l..."
1,kdshapiro,6-Oct,""" kia hyundai most bmw's owners paid for the badge, rather than the car imho. "" kia hyundai i'm glad you added imho. because you d..."
2,fedlawman,6-Oct,""" kia hyundai it's not a matter of "" kia hyundai if "" kia hyundai i can afford a $43,500 car, but is the car "" kia hyundai ..."
3,rockylee,6-Oct,""" kia hyundai under-achieving "" kia hyundai and "" kia hyundai over-achieving "" kia hyundai are relative terms . . . boy i l..."
4,rockylee,6-Oct,"i have 2 friends that own 3 series. i drove both of theirs and agree they are nice car . otoh, like i toyota d they don't do it for m..."


In [4]:
schema_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "empty_or_whitespace": [(df[c].str.strip() == "").sum() for c in df.columns],
    "unique_values": [df[c].nunique() for c in df.columns],
})
display(schema_audit)

date_counts = df.date_label.value_counts().rename_axis("date_label").reset_index(name="posts")
display(date_counts)

,dtype,empty_or_whitespace,unique_values
author,string,0,329
date_label,string,0,44
text,string,9,5986


,date_label,posts
0,7-Mar,1000
1,7-Jan,582
2,6-Nov,460
3,6-Dec,393
4,7-Feb,381
5,6-Oct,357
6,7-Apr,332
7,8-Mar,272
8,11-Oct,241
9,7-Sep,195


**Interpretation.** All three columns parse consistently, but nine text fields are blank after trimming. The highly uneven time distribution matters: one label contributes 1,000 posts, while several contribute only one or two. A top-brand ranking may therefore reflect changes in forum activity over time rather than stable market importance. Final results should include a sensitivity check by period.

## 2 Missingness, duplicates, and contributor concentration

Exact duplicate posts can inflate prevalence and lift. Repeated text posted by different authors or dates is also relevant, so duplicates are measured at both row and text levels. Contributor concentration is reported because a few prolific posters can dominate a conversation corpus.

In [5]:
df["text_trimmed"] = df.text.str.strip()
df["text_normalized_ws"] = df.text.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

dup_summary = pd.DataFrame({
    "definition": [
        "exact duplicate row",
        "duplicate raw text (beyond first)",
        "duplicate whitespace-normalized text (beyond first)",
        "blank text after trimming",
    ],
    "count": [
        int(df.duplicated(["author", "date_label", "text"]).sum()),
        int(df.duplicated("text").sum()),
        int(df.duplicated("text_normalized_ws").sum()),
        int((df.text_trimmed == "").sum()),
    ],
})
display(dup_summary)

duplicate_examples = df[df.duplicated("text_normalized_ws", keep=False)][
    ["author", "date_label", "text_trimmed"]
].sort_values("text_trimmed")
display(duplicate_examples.head(20).assign(text_trimmed=lambda x: x.text_trimmed.str.slice(0, 240)))

,definition,count
0,exact duplicate row,8
1,duplicate raw text (beyond first),14
2,duplicate whitespace-normalized text (beyond first),15
3,blank text after trimming,9


,author,date_label,text_trimmed
1181,dhamilton,6-Dec,
5535,fedlawman,11-Aug,
3879,readerreader,7-Sep,
3878,readerreader,7-Sep,
3876,readerreader,7-Sep,
3211,wale_bate1,7-Apr,
3881,readerreader,7-Sep,
3208,fedlawman,7-Apr,
3209,blueguydotcom,7-Apr,
4941,sweendogy,10-Aug,3500.. not much time spent at that speed due to lots of mass traffic/law enforcement. plus with the new radar my v1 is basically useless...


In [6]:
author_counts = df.author.value_counts()
concentration = pd.DataFrame({
    "group": ["largest author", "top 10 authors", "top 20 authors"],
    "posts": [author_counts.iloc[0], author_counts.iloc[:10].sum(), author_counts.iloc[:20].sum()],
})
concentration["share_of_corpus"] = concentration.posts / len(df)
display(author_counts.head(20).rename_axis("author").reset_index(name="posts"))
display(concentration.assign(share_of_corpus=concentration.share_of_corpus.map("{:.1%}".format)))

,author,posts
0,blueguydotcom,558
1,circlew,359
2,louiswei,328
3,kdshapiro,307
4,fedlawman,278
5,rockylee,253
6,plekto,217
7,shipo,193
8,habitat1,189
9,joe131,132


,group,posts,share_of_corpus
0,largest author,558,9.3%
1,top 10 authors,2814,46.9%
2,top 20 authors,3816,63.6%


**Interpretation.** The largest author alone supplies about 9% of the corpus, the top ten nearly half, and the top twenty nearly two thirds. A primary post-level ranking should therefore be accompanied by an author-balanced sensitivity analysis (for example, count at most one post per author-entity-date label or average entity prevalence within author before aggregating).

## 3 Text length and distribution

Length is measured in characters and regex-tokenized words. Very short posts may lack enough context for semantic attribution; very long posts often contain quoted conversation, travel narratives, or several brand-attribute relationships.

In [7]:
df["char_count"] = df.text.str.len()
df["word_count"] = df.text.str.findall(r"\b[\w$'-]+\b").str.len()

def describe_lengths(s):
    return pd.Series({
        "min": s.min(), "p01": s.quantile(.01), "p05": s.quantile(.05),
        "p25": s.quantile(.25), "median": s.median(), "mean": s.mean(),
        "p75": s.quantile(.75), "p95": s.quantile(.95),
        "p99": s.quantile(.99), "max": s.max(),
    })

length_summary = pd.DataFrame({
    "characters": describe_lengths(df.char_count),
    "words": describe_lengths(df.word_count),
}).round(1)
display(length_summary)
print("Posts under 10 words:", int((df.word_count < 10).sum()))
print("Posts over 500 words:", int((df.word_count > 500).sum()))
display(df.nlargest(5, "word_count")[["author", "date_label", "word_count", "text_trimmed"]].assign(text_trimmed=lambda x: x.text_trimmed.str.slice(0, 400)))

,characters,words
min,1.0,0.0
p01,18.0,3.0
p05,50.0,9.0
p25,170.0,31.0
median,357.0,66.0
mean,545.2,99.6
p75,703.2,129.0
p95,1676.0,306.0
p99,2872.3,508.0
max,6961.0,1279.0


Posts under 10 words: 332
Posts over 500 words: 65


,author,date_label,word_count,text_trimmed
10,rockylee,6-Oct,1279,"rocky, i'm not going to give you a lesson on economics and try to convince you to buy a bmw.thank-you. i guess habitat, i'm partial pal..."
5760,rayainsw,11-Oct,1234,"some details: day 1 &#150; friday 16 sept. &#150; delivery & drive to krimml, austria: we cleared customs and collected our checked bag..."
5761,rayainsw,11-Oct,1221,"day 4 & 5 &#150; monday & tuesday 19, 20 sept. &#150; cinque terre i had decided early on [ for several reasons ] that we would park the..."
5720,markcincinnati,11-Oct,1170,"short answer, no. at this point, an audi audi 2.0t prestige + sline with everything on it except ads would be a great car. i have ads,..."
2239,shipo,7-Mar,1078,"first off, let&#146;s lose the bilge water about what auto rags have written. they far too often stack the deck one way or another for t..."


## 4 Obvious noise and preprocessing artifacts

The counts below are post-level flags. Patterns are diagnostics, not blanket deletion rules. For example, URLs may be analytically irrelevant, but currency amounts are relevant to price discussion. Emoticons may help aspiration interpretation. Quoted text should ideally be separated from the author’s own words rather than removed indiscriminately.

In [8]:
noise_patterns = {
    "URL": r"https?://|www\.",
    "HTML entity": r"&(?:#\d+|[a-z]+);",
    "forum markup": r"\[(?:/?quote|/?url|/?img|/?b|/?i)\b",
    "emoticon": r"(?::[-^]?[()DPp/\\]|;[-^]?[()DPp])",
    "currency amount": r"\$\s?\d",
    "three or more spaces": r" {3,}",
    "non-ASCII character": r"[^\x00-\x7F]",
    "likely broken Toyota insertion": r"\b(?:h\s+toyota\s+ory|soph\s+toyota\s+icated|toyota\s+d|toyota\s+able|toyota\s+mics)\b",
    "Kia-Hyundai inserted pair": r"\bkia\s+hyundai\b",
    "Mercedes-Benz recursive pair": r"\bmercedes-benz\s+benz\b",
}

noise_rows = []
for label, pattern in noise_patterns.items():
    mask = df.text.str.contains(pattern, case=False, regex=True, na=False)
    noise_rows.append({"flag": label, "posts": int(mask.sum()), "share": float(mask.mean())})
noise_audit = pd.DataFrame(noise_rows).sort_values("posts", ascending=False)
display(noise_audit.assign(share=noise_audit.share.map("{:.1%}".format)))

,flag,posts,share
5,three or more spaces,1825,30.4%
8,Kia-Hyundai inserted pair,1697,28.3%
4,currency amount,858,14.3%
7,likely broken Toyota insertion,621,10.3%
3,emoticon,456,7.6%
9,Mercedes-Benz recursive pair,312,5.2%
0,URL,167,2.8%
1,HTML entity,86,1.4%
2,forum markup,2,0.0%
6,non-ASCII character,0,0.0%


In [9]:
artifact_examples = []
for label in ["Kia-Hyundai inserted pair", "Mercedes-Benz recursive pair", "likely broken Toyota insertion"]:
    pattern = noise_patterns[label]
    idxs = df.index[df.text.str.contains(pattern, case=False, regex=True, na=False)][:3]
    for idx in idxs:
        match = re.search(pattern, df.at[idx, "text"], flags=re.I)
        start = max(0, match.start() - 90) if match else 0
        end = min(len(df.at[idx, "text"]), (match.end() + 130) if match else 220)
        artifact_examples.append({
            "flag": label,
            "row": int(idx),
            "context": re.sub(r"\s+", " ", df.at[idx, "text"][start:end]).strip(),
        })
display(pd.DataFrame(artifact_examples))

,flag,row,context
0,Kia-Hyundai inserted pair,0,""" kia hyundai under-achieving "" kia hyundai and "" kia hyundai over-achieving "" kia hyundai are relative terms . . . boy i learned that"
1,Kia-Hyundai inserted pair,1,""" kia hyundai most bmw's owners paid for the badge, rather than the car imho. "" kia hyundai i'm glad you added imho. because you don't s..."
2,Kia-Hyundai inserted pair,2,""" kia hyundai it's not a matter of "" kia hyundai if "" kia hyundai i can afford a $43,500 car, but is the car "" kia hyundai worth "" k"
3,Mercedes-Benz recursive pair,7,"sepower, i'll throw in my hand for the porsche boxster s. if it's price, i'll go with a mercedes-benz benz mercedes-benz benz mercedes-b..."
4,Mercedes-Benz recursive pair,52,eems reasonable for more horsepower and more options.edit - it is true if you configure mercedes-benz benz mercedes-benz benz mercedes-b...
5,Mercedes-Benz recursive pair,59,edit - it is true if you configure mercedes-benz benz mercedes-benz benz mercedes-benz benz mercedes-benz benz mercedes-benz benz merced...
6,likely broken Toyota insertion,4,"ds that own 3 series. i drove both of theirs and agree they are nice car . otoh, like i toyota d they don't do it for me. why ? at that ..."
7,likely broken Toyota insertion,9,n i'm sure. but i will share my personal experience and why i don't ever see financially soph toyota icated buyers flocking from bmw (or...
8,likely broken Toyota insertion,10,t that pal. but i will share my personal experience and why i don't ever see financially soph toyota icated buyers flocking from bmw (or...


### Why this is a blocking validity issue

The strings are consistent with a prior naive entity-replacement pass:

- `kia hyundai` appears as inserted material around quoted phrases rather than only as natural brand discussion.
- `mercedes-benz benz` repeats recursively, sometimes many times in one post.
- `toyota` appears inside broken ordinary words, including contexts corresponding to words such as *history*, *sophisticated*, and *told*.
- Other suspicious fragments include forms such as `honda ing`, `dodge ped`, `pontiac 5`, and `infiniti pontiac`, suggesting overlapping or substring model replacements.

Because the original characters have been overwritten, general cleaning cannot reliably reconstruct the source. Deleting suspicious brand tokens would remove some false positives but would also erase genuine references. The safest course is to obtain the unmodified discussion file. If that is impossible, results must be explicitly labeled as sensitivity analyses under conservative repair rules, not definitive Task A findings.

## 5 Candidate brand and model references

This table is a **discovery audit**, not a final top-ten list. Candidates were surfaced from high-frequency tokens, alphanumeric model-like forms, and automotive contexts in this corpus, then reviewed semantically. No supplied brand/model dictionary was used. Whole-token matching avoids substring matches, but it cannot undo already-corrupted text.

Two metrics are shown: number of posts containing a term (preferred diagnostic) and total raw occurrences. A high occurrences-per-containing-post ratio is a warning sign for quote chains or recursive replacement.

In [10]:
# Corpus-derived candidates retained for transparent inspection.
entity_candidates = [
    ("kia", "brand candidate"), ("hyundai", "brand candidate"),
    ("bmw", "brand candidate"), ("mercedes-benz", "brand candidate"),
    ("toyota", "brand candidate"), ("audi", "brand candidate"),
    ("acura", "brand candidate"), ("honda", "brand candidate"),
    ("cadillac", "brand candidate"), ("lexus", "brand candidate"),
    ("infiniti", "brand candidate"), ("volvo", "brand candidate"),
    ("nissan", "brand candidate"), ("pontiac", "brand candidate"),
    ("lincoln", "brand candidate"), ("saab", "brand candidate"),
    ("subaru", "brand candidate"), ("mazda", "brand candidate"),
    ("jaguar", "brand candidate"), ("porsche", "brand candidate"),
    ("volkswagen", "brand candidate"), ("buick", "brand candidate"),
    ("ford", "brand candidate"), ("chevrolet", "brand candidate"),
    ("bimmer", "brand alias candidate"),
    ("3 series", "model/family candidate"), ("tl", "model candidate"),
    ("cts", "model candidate"), ("is250", "model candidate"),
    ("is350", "model candidate"), ("330i", "model candidate"),
    ("s4", "model candidate"), ("g35", "model candidate"),
    ("c350", "model candidate"), ("a4", "model candidate"),
]

entity_rows = []
for term, candidate_type in entity_candidates:
    pattern = rf"(?<![a-z0-9]){re.escape(term)}(?![a-z0-9])"
    containing = df.text.str.contains(pattern, case=False, regex=True, na=False)
    posts = int(containing.sum())
    occurrences = int(df.text.str.count(pattern, flags=re.I).sum())
    entity_rows.append({
        "candidate": term,
        "candidate_type": candidate_type,
        "posts": posts,
        "post_share": posts / len(df),
        "raw_occurrences": occurrences,
        "occurrences_per_containing_post": occurrences / posts if posts else np.nan,
    })

entity_audit = pd.DataFrame(entity_rows).sort_values(
    ["posts", "raw_occurrences"], ascending=False
).reset_index(drop=True)
display(entity_audit.assign(
    post_share=entity_audit.post_share.map("{:.1%}".format),
    occurrences_per_containing_post=entity_audit.occurrences_per_containing_post.map("{:.2f}".format),
))

,candidate,candidate_type,posts,post_share,raw_occurrences,occurrences_per_containing_post
0,bmw,brand candidate,2025,33.8%,4547,2.25
1,hyundai,brand candidate,1792,29.9%,6189,3.45
2,kia,brand candidate,1710,28.5%,5988,3.50
3,toyota,brand candidate,1680,28.0%,2817,1.68
4,tl,model candidate,782,13.0%,1468,1.88
5,acura,brand candidate,780,13.0%,1611,2.07
6,audi,brand candidate,771,12.8%,1812,2.35
7,honda,brand candidate,714,11.9%,1123,1.57
8,cts,model candidate,642,10.7%,1270,1.98
9,pontiac,brand candidate,631,10.5%,1045,1.66


**Interpretation.** Apparent leaders such as Kia, Hyundai, Toyota, Mercedes-Benz, and several short model-linked brands are contaminated. BMW, Audi, Acura, Lexus, Infiniti, Volvo, Cadillac, and others appear plausible from contextual review, but their final ranking still requires model-to-brand reconciliation on clean text and removal of quoted/duplicate carryover. A token like `TL` is also ambiguous outside automotive context, so short forms require contextual rules or semantic classification.

## 6 Candidate product attributes

Attribute themes were induced from frequent corpus terms and phrases such as *price*, *performance*, *engine*, *handling*, *interior*, *manual transmission*, *resale value*, *fuel economy*, and *rear seat*. The regex codebook below groups surface forms into interpretable themes. These are candidate discussion frequencies, not brand-attributed frequencies: a post can count toward several themes, and quoted text or negation can still create false positives.

In [11]:
attribute_patterns = {
    "price and affordability": r"\b(?:price|prices|pricing|cost|costs|expensive|cheap|afford\w*|money|msrp|discount\w*|lease\w*|payment\w*)\b",
    "performance and acceleration": r"\b(?:performance|accelerat\w*|0\s*[-–]?\s*60|zero\s+to\s+sixty|fast|quick|speed)\w*\b",
    "engine power and torque": r"\b(?:engine\w*|horsepower|\d+\s*hp|power\w*|torque|v6|v8|i6|turbo\w*|supercharg\w*)\b",
    "handling and steering": r"\b(?:handl\w*|steering|corner\w*|roadholding|nimble|agility|agile)\b",
    "interior and materials": r"\b(?:interior|cabin|dashboard|dash|materials?|leather|wood|plastic|fit\s+and\s+finish)\b",
    "ride comfort and noise": r"\b(?:ride|comfort\w*|harsh\w*|smooth\w*|quiet\w*|noise|noisy|refinement|refined)\b",
    "reliability and durability": r"\b(?:reliab\w*|durab\w*|dependab\w*|lasts?|longevity|problems?|repairs?|breaks?|broken)\b",
    "quality and workmanship": r"\b(?:build\s+quality|quality|workmanship|fit\s+and\s+finish|squeak\w*|rattle\w*)\b",
    "fuel economy": r"\b(?:fuel\s+economy|gas\s+mileage|mpg|mileage|fuel|gasoline|gas)\b",
    "drivetrain and traction": r"\b(?:awd|rwd|fwd|all[- ]wheel drive|rear[- ]wheel drive|front[- ]wheel drive|traction)\b",
    "transmission": r"\b(?:transmission\w*|manual\w*|automatic\w*|tranny|gearbox|stick|shift\w*|clutch\w*)\b",
    "styling and design": r"\b(?:style|styling|design\w*|looks?|appearance|beautiful|ugly|attractive|exterior)\b",
    "space seating and practicality": r"\b(?:space|room|roomy|seats?|seating|backseat|rear seat|trunk|cargo|practical\w*|family)\b",
    "features options and technology": r"\b(?:features?|options?|package|navigation|nav|bluetooth|stereo|audio|sound system|sunroof)\b",
    "safety and braking": r"\b(?:safety|safe|airbags?|crash\w*|brakes?|braking|abs|stability control)\b",
    "resale depreciation and value": r"\b(?:resale|residual|depreciat\w*|value|trade[- ]in)\b",
    "maintenance service and warranty": r"\b(?:maintenance|service|dealer\w*|warranty|repairs?)\b",
}

attribute_rows = []
for attribute, pattern in attribute_patterns.items():
    mask = df.text.str.contains(pattern, case=False, regex=True, na=False)
    counts = df.text.str.count(pattern, flags=re.I)
    attribute_rows.append({
        "candidate_attribute": attribute,
        "posts": int(mask.sum()),
        "post_share": float(mask.mean()),
        "raw_occurrences": int(counts.sum()),
    })
attribute_audit = pd.DataFrame(attribute_rows).sort_values("posts", ascending=False)
display(attribute_audit.assign(post_share=attribute_audit.post_share.map("{:.1%}".format)))

,candidate_attribute,posts,post_share,raw_occurrences
0,price and affordability,1609,26.8%,3701
1,performance and acceleration,1264,21.1%,2169
2,engine power and torque,1210,20.2%,2632
6,reliability and durability,1030,17.2%,1739
11,styling and design,987,16.4%,1618
13,features options and technology,875,14.6%,1637
10,transmission,823,13.7%,2084
12,space seating and practicality,765,12.8%,1414
4,interior and materials,741,12.3%,1500
16,maintenance service and warranty,714,11.9%,1538


**Methodological caution.** These patterns deliberately favor recall during discovery. Terms such as *value*, *power*, *room*, *service*, *gas*, *manual*, and *style* can be used non-automotively or refer to another brand in a multi-brand post. Final attribute importance should use post prevalence after semantic validation, merge overlapping themes only when annotators agree, and retain disagreements as an uncertainty measure.

## 7 Edge-case ledger

The following rules identify records that need special handling or deliberate sampling. They are not automatic deletion rules.

In [12]:
edge_masks = {
    "blank text": df.text_trimmed.eq(""),
    "very short (<10 words)": df.word_count.lt(10),
    "very long (>500 words)": df.word_count.gt(500),
    "duplicate normalized text": df.duplicated("text_normalized_ws", keep=False),
    "URL present": df.text.str.contains(noise_patterns["URL"], case=False, regex=True),
    "currency present": df.text.str.contains(noise_patterns["currency amount"], case=False, regex=True),
    "Kia-Hyundai artifact": df.text.str.contains(noise_patterns["Kia-Hyundai inserted pair"], case=False, regex=True),
    "Mercedes recursive artifact": df.text.str.contains(noise_patterns["Mercedes-Benz recursive pair"], case=False, regex=True),
    "Toyota word-fragment artifact": df.text.str.contains(noise_patterns["likely broken Toyota insertion"], case=False, regex=True),
    "many candidate entities (>=5)": pd.Series(False, index=df.index),
}

entity_presence = []
for term, _ in entity_candidates:
    pattern = rf"(?<![a-z0-9]){re.escape(term)}(?![a-z0-9])"
    entity_presence.append(df.text.str.contains(pattern, case=False, regex=True, na=False).astype(int))
edge_masks["many candidate entities (>=5)"] = pd.concat(entity_presence, axis=1).sum(axis=1).ge(5)

edge_summary = pd.DataFrame([
    {"edge_case": label, "posts": int(mask.sum()), "share": float(mask.mean())}
    for label, mask in edge_masks.items()
]).sort_values("posts", ascending=False)
display(edge_summary.assign(share=edge_summary.share.map("{:.1%}".format)))

edge_examples = []
for label, mask in edge_masks.items():
    for idx in df.index[mask][:2]:
        edge_examples.append({
            "edge_case": label,
            "row": int(idx),
            "author": df.at[idx, "author"],
            "words": int(df.at[idx, "word_count"]),
            "text_excerpt": re.sub(r"\s+", " ", df.at[idx, "text"]).strip()[:260],
        })
display(pd.DataFrame(edge_examples))

,edge_case,posts,share
6,Kia-Hyundai artifact,1697,28.3%
9,many candidate entities (>=5),1250,20.8%
5,currency present,858,14.3%
8,Toyota word-fragment artifact,621,10.3%
1,very short (<10 words),332,5.5%
7,Mercedes recursive artifact,312,5.2%
4,URL present,167,2.8%
2,very long (>500 words),65,1.1%
3,duplicate normalized text,23,0.4%
0,blank text,9,0.1%


,edge_case,row,author,words,text_excerpt
0,blank text,1181,dhamilton,0,
1,blank text,3208,fedlawman,0,
2,very short (<10 words),11,rockylee,9,what do you think about the new volvo ?rocky
3,very short (<10 words),38,rockylee,3,"okay, thanxrocky infiniti"
4,very long (>500 words),10,rockylee,1279,"rocky, i'm not going to give you a lesson on economics and try to convince you to buy a bmw.thank-you. i guess habitat, i'm partial pal ..."
5,very long (>500 words),34,rockylee,553,"rocky, you are either crazy or have gone mad. that sentence is probably true with all the non-m bmw but i honestly don't think the cts-v..."
6,duplicate normalized text,1181,dhamilton,0,
7,duplicate normalized text,1853,qbrozen,81,"i haven't tried the pontiac 5's system, but i had those bose system in my 350z and it wasn't very good at all. and, by all accounts i've..."
8,URL present,37,rockylee,220,"dude, the sport suspension package is the top package and the only stand-alone option is a $2k+ pre-collision.i'm not what source you ar..."
9,URL present,40,louiswei,20,i'm not what source you are using the source i used is call www.lexus.comnow i rest my case...


## 8 Defensible Task A pipeline

### Phase 1 Recover and preserve source

1. Obtain the original unmodified Edmunds discussion data or the exact transformation code and mapping.
2. Hash and retain the source; never overwrite text.
3. Create derived fields for whitespace normalization, quote segmentation, URL handling, and duplicate flags.
4. Keep an exclusion ledger with a reason for every removed post.

### Phase 2 Dictionary-free candidate discovery

1. Rank unigrams, bigrams, and alphanumeric strings by **document frequency**, not token frequency.
2. Use contextual windows around terms such as *drive*, *own*, *buy*, *dealer*, *sedan*, and *model* to surface automotive entities.
3. Add unsupervised phrase extraction and named-entity/embedding clustering to consolidate spelling variants.
4. Surface product attributes from frequent noun phrases and contextual clusters; retain the original phrases as evidence.

### Phase 3 Semantic reconciliation with an LLM

Send bounded post excerpts plus discovered candidates, not a preloaded brand dictionary. Require strict JSON with:

- canonical brand;
- surface form and character span;
- reference type: brand, model, abbreviation, nickname, or ambiguous;
- whether the reference is asserted by the author or appears only in quoted text;
- product attribute and the brand it modifies, if any;
- evidence excerpt;
- confidence and ambiguity reason.

The LLM may propose a canonicalization, but a transparent alias table is created only after repeated evidence and human review. Short or ambiguous forms must be resolved from context rather than globally replaced.

### Phase 4 Define importance and validate

Primary ranking: unique substantive posts mentioning the canonical brand divided by eligible posts. Secondary sensitivity analyses: raw mention count, unique-author prevalence, author-balanced prevalence, and time-stratified prevalence. Report a brand only when lexical and semantic evidence support it.

Manually label a stratified sample of at least 100 records, oversampling ambiguous short forms, multi-brand posts, quoted passages, rare candidates, and disagreement between NLP and LLM. Report precision/recall/F1 for entity presence, accuracy for canonical brand, agreement for attribute themes, and an error taxonomy. Do not tune and evaluate on the same sample; use a separate adjudicated holdout where feasible.

## 9 Toy-data checks required before scaling

The rubric requires manually verifiable toy examples. The next cell tests the lexical mechanics that later Tasks A and B will rely on: word boundaries, alias reconciliation, post-level counting, duplicate policy, and co-occurrence. This is not a test of semantic correctness.

In [13]:
toy = pd.DataFrame({"text": [
    "I drove a BMW and an Audi.",
    "The 3 Series was fun; the BMW badge is expensive.",
    "Audio quality was good.",  # must not match Audi
    "I own an A4.",
    "I drove a BMW and an Audi.",  # exact duplicate
]})

toy_aliases = {
    "bmw": [r"\bbmw\b", r"\b3\s+series\b"],
    "audi": [r"\baudi\b", r"\ba4\b"],
}

toy_unique = toy.drop_duplicates("text").copy()
for brand, patterns in toy_aliases.items():
    combined = "(?:" + "|".join(patterns) + ")"
    toy_unique[brand] = toy_unique.text.str.contains(combined, case=False, regex=True)

assert toy_unique.bmw.sum() == 2
assert toy_unique.audi.sum() == 2
assert not toy_unique.loc[toy_unique.text.str.startswith("Audio"), "audi"].iloc[0]
assert (toy_unique.bmw & toy_unique.audi).sum() == 1

display(toy_unique)
print("Toy checks passed: boundaries, aliases, deduplication, and post-level co-occurrence.")

,text,bmw,audi
0,I drove a BMW and an Audi.,True,True
1,The 3 Series was fun; the BMW badge is expensive.,True,False
2,Audio quality was good.,False,False
3,I own an A4.,False,True


Toy checks passed: boundaries, aliases, deduplication, and post-level co-occurrence.


### Discovery and validation informed by the other decks

For Task A, add TF-IDF and cosine similarity for retrieving related contexts and near-duplicate candidates (September 14 slides 4–11). Preserve model identifiers, numbers, negation, and phrases such as “3 series”; generic stemming must not damage entity strings. Embeddings can suggest aliases, but semantic similarity is not proof that two names denote the same brand. Human-reviewed evidence remains necessary.

The September 21 deck provides two complementary discovery methods. TF-IDF plus document clustering can expose major conversation groups; count-based LDA can expose several themes within one post. Compare topic counts and random seeds, inspect top words and representative posts, and have the LLM suggest labels with quoted evidence. A topic can mix an attribute, a brand, and forum chatter, so its probability is not an attribute mention frequency. Keep multiple attribute labels per post. These methods are specified for the next discovery phase; they have not been fitted in this update.

The earlier attribute table is a manually reconciled lexical candidate codebook. It does not by itself complete the advanced NLP discovery requirement. The final submission needs executed discovery outputs, a frozen alias/codebook version, and evaluation on a held-out sample. Include a representative random sample as well as difficult cases; report challenge-set errors separately so oversampling ambiguity does not distort aggregate accuracy. Purity, entropy, and Rand index from the clustering lecture apply only if suitable reference class labels exist.

## 11 Audit findings and immediate decision

### Reliable findings

- 6,000 rows and three consistently parsed columns.
- 329 authors, 44 date labels, and 5,986 distinct raw text values.
- Blank posts, exact duplicates, whitespace-normalized duplicates, extreme post lengths, and strong author/time concentration are present and measurable.
- The corpus contains substantive discussion of price, performance, power, styling, reliability, transmission, space, features, handling, interior, and related automobile attributes.

### Findings that should not yet be treated as final

- The ten most important brands.
- Frequencies for brands affected by model/substring replacement.
- Attribute-to-brand attribution in multi-brand or quoted posts.

### Immediate decision

Pause final Task A ranking and request the pristine discussion file. Once obtained, rerun this audit, execute the dictionary-free candidate discovery, apply contextual entity reconciliation, conduct the stratified manual validation, and only then freeze the top-ten brand and attribute tables that feed Tasks B through G.

# Task B — Establish a simple NLP-based competitive baseline

**Required:** Calculate the full lift matrix for Task A's ten brands using transparent lexical/NLP rules and a consistent unit of analysis.

**Current work:** Tested arithmetic and selected brand-pair diagnostics. These examples do not constitute the complete required matrix.

## 13 Lift as taught in Session 2

For a fixed universe of eligible posts, let N be its size, n(A) the number of posts mentioning A, n(B) the number mentioning B, and n(A,B) the number mentioning both. Count each entity at most once per post.

\[
\operatorname{Lift}(A,B)=\frac{P(A\cap B)}{P(A)P(B)}
=\frac{N\,n(A,B)}{n(A)n(B)}.
\]

Lift 1 means the observed co-occurrence equals the independence benchmark; lift above 1 means more co-occurrence than expected; below 1 means less. A lift of 2 means twice the expected joint prevalence, not twice the market share or a causal effect. Session 2 slide 6 groups values at or below 1 as lacking positive association; we retain the original numerical values for auditability.

**Choice and alternative.** Use post-level binary indicators because repetition should not multiply a post's contribution. Raw token counts overweight long posts and recursive substitutions. TF-IDF supports discovery and text similarity but is not a probability of a post-level event and therefore does not replace these counts in lift.

**Denominator.** Primary diagnostic universe: nonblank posts after exact duplicate-row removal, keeping distinct authors/dates and all posts without selected entities. Dropping every post without a top-ten brand would condition the analysis on that subset and change lift. Global identical-text removal is a sensitivity case because short repeated replies from different users can be genuine. All NLP-versus-LLM comparisons must use the same eligible post IDs.

**Support.** Report N, both marginal counts, joint count, expected joint count, and lift together. Large lift from a few observations is unstable. A joint-count threshold can flag weak support but does not prove statistical significance. Later uncertainty estimates should resample authors or other suitable groups to account for dependence among posts.

In [14]:
def lift_from_counts(N, n_a, n_b, n_ab):
    assert 0 <= n_a <= N and 0 <= n_b <= N
    assert max(0, n_a + n_b - N) <= n_ab <= min(n_a, n_b)
    return N * n_ab / (n_a * n_b) if n_a and n_b else np.nan

# Session 2 slide 8: these are lecture numbers, not counts from sample_data.csv.
lecture_example = pd.DataFrame([
    {'model': 'Toyota Camry', 'N': 868174, 'model_posts': 34559, 'safety_posts': 4534, 'joint_posts': 379},
    {'model': 'Volvo S40', 'N': 868174, 'model_posts': 2320, 'safety_posts': 4534, 'joint_posts': 120},
])
lecture_example['lift'] = lecture_example.apply(
    lambda r: lift_from_counts(r.N, r.model_posts, r.safety_posts, r.joint_posts), axis=1)
display(lecture_example.round(4))

# Manually checkable examples: expected count is 20*8/100 = 1.6;
# 4 actual co-occurrences gives 4/1.6 = 2.5.
assert lift_from_counts(100, 20, 8, 4) == 2.5
assert lift_from_counts(100, 20, 10, 2) == 1
assert lift_from_counts(100, 20, 10, 0) == 0
assert np.isnan(lift_from_counts(100, 0, 10, 0))
assert lift_from_counts(100, 8, 20, 4) == lift_from_counts(100, 20, 8, 4)
# Same four-post toy universe used earlier: BMW=2, Audi=2, both=1.
assert lift_from_counts(len(toy_unique), int(toy_unique.bmw.sum()),
                        int(toy_unique.audi.sum()), int((toy_unique.bmw & toy_unique.audi).sum())) == 1
# Session 2 slide 27: conditional universe contains 500 performance posts.
assert np.isclose(lift_from_counts(500, 250, 300, 200), 4/3)
print('Lift checks passed: association, independence, zero joint count, absent entity, symmetry, and conditional lift.')

,model,N,model_posts,safety_posts,joint_posts,lift
0,Toyota Camry,868174,34559,4534,379,2.0999
1,Volvo S40,868174,2320,4534,120,9.9042


Lift checks passed: association, independence, zero joint count, absent entity, symmetry, and conditional lift.


## 14 Applied lift diagnostic on the supplied corpus

The pair selection below is deliberate: Kia–Hyundai tests the inserted-pair anomaly, while BMW–Audi and BMW–Lexus provide automotive comparisons. These are explicit-token diagnostics, not the final reconciled ten-brand matrix required in Task B. Exact matches cannot recover model names overwritten in the supplied text.

For brand–attribute analysis, a shared post is the transparent baseline. It does not establish that the attribute describes that brand. Task E must separately inspect attributable evidence, and Task C must retain consistent counting rules when LLM interpretation changes which references count.

In [15]:
raw_columns = ['author', 'date_label', 'text']
eligible = df.loc[df.text_trimmed.ne('')].drop_duplicates(raw_columns).copy()

def presence(frame, term):
    return frame.text.str.contains(
        rf'(?<![a-z0-9]){re.escape(term)}(?![a-z0-9])', case=False, regex=True, na=False).to_numpy(dtype=bool)

def association_row(label, a, b, universe):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    N = len(a)
    assert N == len(b)
    n_a, n_b, n_ab = int(a.sum()), int(b.sum()), int((a & b).sum())
    return {'universe': universe, 'pair': label, 'N': N, 'n_A': n_a, 'n_B': n_b,
            'n_AB': n_ab, 'expected_AB': n_a*n_b/N if N else np.nan,
            'lift': lift_from_counts(N, n_a, n_b, n_ab)}

diagnostics = []
universes = {
    'all raw rows': df,
    'nonblank, exact row deduplication': eligible,
    'nonblank, normalized text deduplication': eligible.drop_duplicates('text_normalized_ws'),
}
for label, frame in universes.items():
    for a, b in [('kia', 'hyundai'), ('bmw', 'audi'), ('bmw', 'lexus')]:
        diagnostics.append(association_row(f'{a} / {b}', presence(frame, a), presence(frame, b), label))
lift_diagnostics = pd.DataFrame(diagnostics)
display(lift_diagnostics.round(4))

# Repeated occurrences within a post do not change its binary event.
probe = pd.DataFrame({'text': ['BMW Audi', 'BMW BMW Audi Audi']})
assert presence(probe, 'bmw').all() and presence(probe, 'audi').all()

,universe,pair,N,n_A,n_B,n_AB,expected_AB,lift
0,all raw rows,kia / hyundai,6000,1710,1792,1701,510.7200,3.3306
1,all raw rows,bmw / audi,6000,2025,771,411,260.2125,1.5795
2,all raw rows,bmw / lexus,6000,2025,616,374,207.9000,1.7989
3,"nonblank, exact row deduplication",kia / hyundai,5986,1710,1792,1701,511.9145,3.3228
4,"nonblank, exact row deduplication",bmw / audi,5986,2025,771,411,260.8211,1.5758
5,"nonblank, exact row deduplication",bmw / lexus,5986,2025,615,374,208.0479,1.7977
6,"nonblank, normalized text deduplication",kia / hyundai,5984,1710,1792,1701,512.0856,3.3217
7,"nonblank, normalized text deduplication",bmw / audi,5984,2025,771,411,260.9083,1.5753
8,"nonblank, normalized text deduplication",bmw / lexus,5984,2025,615,374,208.1175,1.7971


# Task C — Recalculate lift using LLM semantic attribution and reconcile the results

**Required:** Run contextual brand extraction, calculate a second lift matrix, quantify agreement and disagreement with Task B, and explain the largest differences using source posts.

**Status: Not executed.** No corpus-wide LLM output or semantic lift matrix is present. Keyword results must not be presented as LLM results.

Save evidence excerpts, model identifiers, prompts and usage for actual extraction calls. Review a representative sample plus difficult cases and report errors. Use the same eligible post IDs and ten brands in both matrices.

### Comparison method
Keep post eligibility and entity set fixed and compare the 45 unique off-diagonal pairs for ten brands. Report absolute lift differences, rank changes, Pearson correlation, and Spearman correlation, as adapted from Session 2 slide 18. Correlation alone can hide material level changes, so examine the largest disagreements with source excerpts. Calculate arithmetic in Python for both methods. Log model/version, prompts, call counts, tokens, and costs for semantic API runs.

No external LLM API calls were made in this update. The assistant did interpret the lecture material and corpus examples while authoring the notebook; those chat-level tokens and costs were not available for this audit. The zero ledger above refers only to notebook/API extraction, not to the absence of all AI assistance. A complete submission should distinguish these two uses.

## 10 LLM audit log

No external LLM classification or extraction call is made by this notebook. The current row therefore reports zero usage rather than inventing token or cost figures. When semantic extraction is run, append one row per batch and retain raw responses, validation status, and pricing assumptions.

In [16]:
llm_log = pd.DataFrame([{
    "step": "Task A audit notebook",
    "model": "none",
    "api_calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "status": "No semantic API classification executed yet",
}])
display(llm_log)

,step,model,api_calls,input_tokens,output_tokens,total_tokens,estimated_cost_usd,status
0,Task A audit notebook,none,0,0,0,0,0.0,No semantic API classification executed yet


# Task D — Map the competitive structure

**Required:** Produce an MDS map using a justified transformation of the preferred association measure, add clustering, interpret competitive relationships, and assess sensitivity to the NLP-versus-LLM choice.

**Status: Not executed.** The method below is a plan, not a completed map or cluster solution.

### MDS and clustering method
Follow Session 2 slides 11 and 19–21: for positive off-diagonal lifts use dissimilarity d(A,B)=1/Lift(A,B), set the diagonal to zero, and run MDS to approximate the full distances in two dimensions. A zero lift has no finite inverse: disclose the chosen cap or coherent smoothing rule and test sensitivity instead of silently dividing by zero. Inverse lift need not satisfy the triangle inequality; inspect MDS stress and distance fit before interpreting the map. MDS axes have no intrinsic attribute labels.

Use average-linkage hierarchical clustering on the full dissimilarity matrix, or k-medoids as an alternative. Compare plausible cluster counts with silhouette scores and interpretability. Overlay cluster labels on the MDS map. The document-clustering lecture's k-means example concerns document vectors; Session 2 specifically discourages clustering brands using k-means on compressed two-dimensional map coordinates.

# Task E — Analyze brand–attribute positioning

**Required:** Select five important brands and five important attributes from the earlier analysis. Calculate justified associations and establish which attributes refer to which brands in multi-brand posts.

**Current work:** Six lexical co-occurrence examples for two brands and three attributes. A shared post does not by itself establish a brand–attribute relationship. The required five-by-five analysis and semantic validation remain unfinished.

In [17]:
attribute_diagnostics = []
for brand in ['bmw', 'audi']:
    for attribute in ['price and affordability', 'handling and steering', 'interior and materials']:
        flag = eligible.text.str.contains(attribute_patterns[attribute], case=False, regex=True, na=False)
        attribute_diagnostics.append(association_row(
            f'{brand} / {attribute}', presence(eligible, brand), flag, 'nonblank, exact row deduplication'))
display(pd.DataFrame(attribute_diagnostics).round(4))
print('All displayed associations are lexical diagnostics pending semantic validation.')

,universe,pair,N,n_A,n_B,n_AB,expected_AB,lift
0,"nonblank, exact row deduplication",bmw / price and affordability,5986,2025,1608,838,543.9693,1.5405
1,"nonblank, exact row deduplication",bmw / handling and steering,5986,2025,694,349,234.7728,1.4865
2,"nonblank, exact row deduplication",bmw / interior and materials,5986,2025,741,349,250.6724,1.3923
3,"nonblank, exact row deduplication",audi / price and affordability,5986,771,1608,322,207.1113,1.5547
4,"nonblank, exact row deduplication",audi / handling and steering,5986,771,694,174,89.3876,1.9466
5,"nonblank, exact row deduplication",audi / interior and materials,5986,771,741,203,95.4412,2.1270


All displayed associations are lexical diagnostics pending semantic validation.


## 15 Conditional lift and course-aligned next steps

Session 2 slides 26–27 ask whether Audi and BMW are associated within discussion of performance. For attribute P:

\[
\operatorname{Lift}(A,B\mid P)=
\frac{n(P)\,n(A,B,P)}{n(A,P)n(B,P)}.
\]

This calculation restricts the universe to posts mentioning P. It is a different question from ordinary brand–attribute lift. A post saying “Audi has good handling, but BMW is expensive” contains multiple entities and attributes; assigning every attribute to both brands overstates positioning. Preserve mention presence and relationship attribution as separate fields. Semantic pair-level relations should be reported separately from binary-presence lift if the relationship rule goes beyond joint presence.

In [18]:
performance_mask = eligible.text.str.contains(
    attribute_patterns['performance and acceleration'], case=False, regex=True, na=False)
performance_posts = eligible.loc[performance_mask]
conditional_diagnostic = association_row(
    'Audi / BMW conditional on lexical performance theme',
    presence(performance_posts, 'audi'), presence(performance_posts, 'bmw'),
    'performance-theme posts within eligible universe')
display(pd.DataFrame([conditional_diagnostic]).round(4))

,universe,pair,N,n_A,n_B,n_AB,expected_AB,lift
0,performance-theme posts within eligible universe,Audi / BMW conditional on lexical performance theme,1262,254,609,169,122.5721,1.3788


# Task F — Identify the most aspirational brand

**Required:** Define aspiration as an actual desire to buy or own a brand, operationalize it from context, validate classifications, report the winning brand, and discuss business implications.

**Status: Not executed. No aspiration winner has been established.**

Proposed definition: count a post when its author expresses a personal, affirmative desire or intention to acquire or own a vehicle. Distinguish a concrete intention from a conditional wish and report both. Exclude current ownership alone, praise alone, negated intentions and another person's quoted desire. Resolve the target brand from evidence rather than assuming that the nearest brand is the intended target.

Report counts and unique authors, plus the share of brand-referencing posts expressing aspiration. Inspect whether the leader changes when using counts versus rates, with and without conditional wishes. Manually inspect positive predictions and sampled negatives, recording error types. State the denominator for every reported rate.

# Task G — Advise the client

**Required:** Combine competitive, positioning and aspiration results into a small number of actionable recommendations for JD Power and its clients. Distinguish robust conclusions from those sensitive to analytical choices.

**Status: Final recommendations pending the preceding analyses.**

For every final recommendation, name the client action, supporting result and task, robustness evidence, and limitation. Examples of decisions this analysis can inform include which brands to include in a comparison set, which attributes to investigate in positioning research, and which aspirational audiences warrant follow-up research. These are possible uses, not findings established by this draft.

The supported immediate recommendation is to investigate the apparent replacement artifacts and validate entity extraction before relying on brand rankings. Conversation frequency does not establish market share, and aspiration does not establish purchase conversion.

## Course methods and source references

## 12 Course slides and revisions to the analytical approach

The assignment document remains the governing rubric. The four supplied lecture decks inform the methods; classroom exercises and final-project instructions do not add requirements to this assignment. “Lift” is the association measure used here.

| Course material | Relevant slides | Application to this notebook and subsequent tasks |
|---|---|---|
| Session_2_MSBTAI_F2026_Aug31.pptx | 3–8 | Discover entities and attributes, count post-level occurrences, calculate lift relative to independence. |
| Same Session 2 deck | 11–12, 22 | NLP discovers candidates; LLM interprets contextual meaning; Python performs arithmetic. |
| Same Session 2 deck | 18–21 | Compare unique brand pairs with Pearson and Spearman correlation; transform lift into dissimilarity; cluster the full matrix with average-linkage hierarchical clustering or k-medoids. |
| Same Session 2 deck | 26–27 | Conditional lift examines whether two brands associate within an attribute discussion. |
| MSBT&AI_Session 3 Sep 14 2026.pptx | 4–11, 22–28, 44 | Binary presence for counting; TF-IDF and cosine for contextual similarity; embeddings propose semantically related forms. |
| MSBT&AI September 21 2026 Session 3 Clustering Topic Modeling.pptx | 3–4, 6–15 | TF-IDF plus document clustering and count-based LDA offer additional attribute discovery; posts can discuss multiple topics. The title slide calls this Session 4. |
| Same September 21 deck | 16–19 | Purity, entropy, and Rand index require external reference labels; do not invent them for unlabeled data. |
| Session1_UDAGenAI_F2026_MSBTAI.pptx | 4, 7–10, 17 | Connect measurements to competitive questions and acknowledge forum sampling limitations. |

The corpus audit supports continuing with calculations and sensitivity analysis even before the original text becomes available. Its frequencies describe this supplied file. Authentic consumer brand references require additional validation because apparent replacement artifacts can create artificial associations. The origin of the damage is a hypothesis; it cannot be established without source data or transformation history.

## Review decisions and remaining work

Use this space to record our decisions and the supporting evidence.

| Decision | Current position | Review notes |
|---|---|---|
| Source text and apparent replacement artifacts | Preserve raw data; seek provenance; quantify sensitivity | To discuss |
| Unit and eligibility | Post-level presence; nonblank and exact-row deduplication for lift diagnostics | To confirm |
| Important brands | Rank validated canonical brand prevalence; inspect author/time sensitivity | Not finalized |
| Attributes | Current lexical themes are candidate categories | Validate phrases and overlap |
| Task C LLM extraction | Preserve evidence spans, unresolved cases, model and cost records | Not run |
| Task D map and clusters | Inverse-lift distances with explicit zero handling; cluster full distances | Not run |
| Task E relationships | Check which brand each attribute describes | Not run at required scope |
| Task F aspiration | Distinguish desire to buy/own from ownership, negation, quoted wishes and general praise | Define and validate |
| Task G advice | Build recommendations from completed evidence and sensitivity results | Pending |

The existing outputs do not include a validated final top ten, a full NLP matrix, a semantic matrix, an MDS map, final clusters, a five-by-five attributed positioning analysis, or an aspiration winner. These are the next assignment deliverables, not claims made by this review notebook.